# 🛡️ Data Encryption Standard (DES) from Scratch
### Mata Kuliah: MA4151 - Kriptografi
**Berbasis Teks Karakter Asli, Angka Desimal (0-255), dan Bit Biner (0/1)**

---

## 🎯 Tujuan Pembelajaran
1. Memahami bahwa **64 bit = 8 karakter teks** (misal kata `"KOMPUTER"`).
2. Mempelajari alur konversi: **Karakter $\to$ Nilai Desimal (ASCII) $\to$ 8 Bit Biner**.
3. Menguasai fungsi permutasi dan manipulasi bit DES:
   - Initial Permutation ($IP$) dan Inverse Initial Permutation ($IP^{-1}$)
   - Key Schedule ($PC\text{-}1$, pergeseran kiri melingkar, $PC\text{-}2$)
   - Fungsi Feistel $f(R, K)$ (Ekspansi $E$, XOR subkunci, 8 Kotak-S, Permutasi $P$)
   - 16 Ronde Feistel Enkripsi dan Dekripsi
4. Menjalankan eksperimen **Efek Salju (Avalanche Effect)** pada teks nyata.


---
## 1. Konversi Teks $\to$ Desimal $\to$ Bit Biner (Bit Helpers)
Komputer menyimpan huruf sebagai angka desimal standar (ASCII $0-255$). Setiap angka desimal disimpan sebagai deretan 8 bit biner ($0$ dan $1$).


In [ ]:
def text_to_bits(text: str) -> list[int]:
    """Mengonversi teks string menjadi daftar bit biner (0 dan 1)."""
    data_bytes = text.encode('ascii')
    bits = []
    for b in data_bytes:
        for i in range(7, -1, -1):
            bits.append((b >> i) & 1)
    return bits

def bits_to_decimals(bits: list[int]) -> list[int]:
    """Mengonversi daftar bit biner menjadi daftar angka desimal (0-255)."""
    decimals = []
    for i in range(0, len(bits), 8):
        val = 0
        for b in bits[i:i+8]:
            val = (val << 1) | b
        decimals.append(val)
    return decimals

def permute(bits: list[int], table: list[int]) -> list[int]:
    """Menerapkan tabel permutasi (1-indexed) pada daftar bit."""
    return [bits[idx - 1] for idx in table]

# Contoh Konversi Teks Nyata
teks_contoh = "KOMPUTER" # 8 karakter = 64 bit
bits_contoh = text_to_bits(teks_contoh)
desimal_contoh = bits_to_decimals(bits_contoh)

print(f"Teks Asli        : {teks_contoh} (Panjang: {len(teks_contoh)} huruf)")
print(f"Nilai Desimal    : {desimal_contoh}")
print(f"Bit Biner (64-bit): {' '.join(''.join(map(str, bits_contoh[i:i+8])) for i in range(0, 64, 8))}")


---
## 2. Tabel Permutasi Baku DES (Standar FIPS 46-3)


In [ ]:
IP = [
    58, 50, 42, 34, 26, 18, 10, 2,
    60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6,
    64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17, 9, 1,
    59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5,
    63, 55, 47, 39, 31, 23, 15, 7
]

IP_INV = [
    40, 8, 48, 16, 56, 24, 64, 32,
    39, 7, 47, 15, 55, 23, 63, 31,
    38, 6, 46, 14, 54, 22, 62, 30,
    37, 5, 45, 13, 53, 21, 61, 29,
    36, 4, 44, 12, 52, 20, 60, 28,
    35, 3, 43, 11, 51, 19, 59, 27,
    34, 2, 42, 10, 50, 18, 58, 26,
    33, 1, 41, 9, 49, 17, 57, 25
]

PC1 = [
    57, 49, 41, 33, 25, 17, 9,
    1, 58, 50, 42, 34, 26, 18,
    10, 2, 59, 51, 43, 35, 27,
    19, 11, 3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15,
    7, 62, 54, 46, 38, 30, 22,
    14, 6, 61, 53, 45, 37, 29,
    21, 13, 5, 28, 20, 12, 4
]

SHIFTS = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

PC2 = [
    14, 17, 11, 24, 1, 5,
    3, 28, 15, 6, 21, 10,
    23, 19, 12, 4, 26, 8,
    16, 7, 27, 20, 13, 2,
    41, 52, 31, 37, 47, 55,
    30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53,
    46, 42, 50, 36, 29, 32
]

E_TABLE = [
    32, 1, 2, 3, 4, 5,
    4, 5, 6, 7, 8, 9,
    8, 9, 10, 11, 12, 13,
    12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21,
    20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29,
    28, 29, 30, 31, 32, 1
]

S_BOXES = [
    # S1
    [
        [14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
        [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
        [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
        [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13]
    ],
    # S2
    [
        [15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
        [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
        [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
        [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9]
    ],
    # S3
    [
        [10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
        [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
        [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
        [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12]
    ],
    # S4
    [
        [7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
        [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
        [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
        [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14]
    ],
    # S5
    [
        [2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
        [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
        [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
        [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3]
    ],
    # S6
    [
        [12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
        [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
        [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
        [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13]
    ],
    # S7
    [
        [4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
        [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
        [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
        [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12]
    ],
    # S8
    [
        [13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
        [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
        [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
        [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11]
    ]
]

P_TABLE = [
    16, 7, 20, 21, 29, 12, 28, 17,
    1, 15, 23, 26, 5, 18, 31, 10,
    2, 8, 24, 14, 32, 27, 3, 9,
    19, 13, 30, 6, 22, 11, 4, 25
]

print("Tabel standar DES siap!")


---
## 3. Pembangkitan Kunci Ronde & Fungsi Feistel $f(R, K)$


In [ ]:
def des_generate_keys(key_bits: list[int]) -> list[list[int]]:
    permuted_key = permute(key_bits, PC1)
    c = permuted_key[:28]
    d = permuted_key[28:]
    subkeys = []
    for shift in SHIFTS:
        c = c[shift:] + c[:shift]
        d = d[shift:] + d[:shift]
        subkeys.append(permute(c + d, PC2))
    return subkeys

def des_f(r: list[int], subkey: list[int]) -> list[int]:
    expanded = permute(r, E_TABLE)
    xored = [bit ^ k for bit, k in zip(expanded, subkey)]
    sbox_out = []
    for i in range(8):
        chunk = xored[i * 6 : (i + 1) * 6]
        row = (chunk[0] << 1) | chunk[5]
        col = (chunk[1] << 3) | (chunk[2] << 2) | (chunk[3] << 1) | chunk[4]
        val = S_BOXES[i][row][col]
        for shift in (3, 2, 1, 0):
            sbox_out.append((val >> shift) & 1)
    return permute(sbox_out, P_TABLE)

def des_encrypt_block(pt_bits: list[int], subkeys: list[list[int]], verbose: bool = False) -> list[int]:
    bits = permute(pt_bits, IP)
    l, r = bits[:32], bits[32:]
    for round_idx, k in enumerate(subkeys, 1):
        f_res = des_f(r, k)
        l, r = r, [l_bit ^ f_bit for l_bit, f_bit in zip(l, f_res)]
        if verbose:
            print(f"Ronde {round_idx:2d}: L (desimal)={bits_to_decimals(l)}, R (desimal)={bits_to_decimals(r)}")
    return permute(r + l, IP_INV)

def des_decrypt_block(ct_bits: list[int], subkeys: list[list[int]]) -> list[int]:
    return des_encrypt_block(ct_bits, subkeys[::-1])


---
## 4. Eksekusi Enkripsi Teks Nyata: `"KOMPUTER"` dengan Kunci `"RAHASIA1"`


In [ ]:
pesan_teks = "KOMPUTER"
kunci_teks = "RAHASIA1"

pt_bits = text_to_bits(pesan_teks)
k_bits = text_to_bits(kunci_teks)
subkeys = des_generate_keys(k_bits)

print("=== ENKRIPSI DES ===")
print(f"Pesan Asli   : '{pesan_teks}' -> Desimal: {bits_to_decimals(pt_bits)}")
print(f"Kunci Rahasia: '{kunci_teks}' -> Desimal: {bits_to_decimals(k_bits)}\n")

print("--- Putaran 16 Ronde Feistel (Nilai Desimal Tiap 4 Byte) ---")
ct_bits = des_encrypt_block(pt_bits, subkeys, verbose=True)
ct_decimals = bits_to_decimals(ct_bits)

print("\n=== HASIL AKHIR ===")
print(f"Ciphertext Desimal (8 Byte): {ct_decimals}")
print(f"Ciphertext Biner (64 Bit)  : {' '.join(''.join(map(str, ct_bits[i:i+8])) for i in range(0, 64, 8))}")

# Dekripsi
dec_bits = des_decrypt_block(ct_bits, subkeys)
dec_text = bytes(bits_to_decimals(dec_bits)).decode('ascii')
print(f"\nHasil Dekripsi Kembali     : '{dec_text}' (100% Cocok!)")


---
## 5. Eksperimen Efek Salju (Avalanche Effect)


In [ ]:
import matplotlib.pyplot as plt

def trace_des_round_bits(pt_bits, subkeys):
    bits = permute(pt_bits, IP)
    l, r = bits[:32], bits[32:]
    history = [l + r]
    for k in subkeys:
        f_res = des_f(r, k)
        l, r = r, [l_bit ^ f_bit for l_bit, f_bit in zip(l, f_res)]
        history.append(l + r)
    return history

pt1 = text_to_bits("KOMPUTER")
pt2 = text_to_bits("KOMPUTES") # 'R' (82) -> 'S' (83), beda tepat 1 bit!

h1 = trace_des_round_bits(pt1, subkeys)
h2 = trace_des_round_bits(pt2, subkeys)
diffs = [sum(b1 != b2 for b1, b2 in zip(s1, s2)) for s1, s2 in zip(h1, h2)]

print("Jumlah Bit Berubah di Setiap Ronde (dari 64 bit):")
for r, d in enumerate(diffs):
    print(f"Ronde {r:2d}: {d:2d} bit berubah ({(d/64)*100:5.1f}%)")

plt.figure(figsize=(9, 4.5))
plt.plot(range(17), diffs, marker='o', color='#1f77b4', linewidth=2, label='Bit Berubah (DES)')
plt.axhline(y=32, color='red', linestyle='--', label='Ideal SAC (50% = 32 bit)')
plt.title("Analisis Efek Salju (Avalanche Effect) pada DES", fontsize=12, fontweight='bold')
plt.xlabel("Ronde Feistel")
plt.ylabel("Jumlah Bit Berubah (dari 64 bit)")
plt.xticks(range(17))
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()
